Test which ISs are completely covered by reads

# Create input data
## fasta

In [1]:
import pandas as pd
import re, io, subprocess
import os, sys

from Bio import SeqIO
from Bio.Seq import Seq
import pysam

In [2]:
file_list_path = '../data/File_list_20250204.csv'
is_pos_path = '../data/IS_positions.csv'
ref_fasta_dir = '../tmp/clean_fasta/'
bat_dir = '../exp/IGV_script/IS_cover/'
bed_dir = '../exp/dat/bed/'
win_base_dir = r'V:\2022\analysis_2023\20250112_Revise\for_publication'
igv_snapshot_dir = win_base_dir + r'\exp\fig\IGV\IS_cover'
igv_fasta_dir = win_base_dir + r'\tmp\clean_fasta'
igv_bam_dir = win_base_dir + r'\exp\bam\mm2_trimmed\all'
igv_bed_dir = win_base_dir + r'\exp\dat\bed'

os.makedirs(bat_dir, exist_ok=True)
os.makedirs(bed_dir, exist_ok=True)
igv_snapshot_dir_ = '../exp/fig/IGV/IS_cover'
os.makedirs(igv_snapshot_dir_, exist_ok=True)

# trimmed (but not cut-adapted reads), all reads including those clipped
mm2_output_dir = os.path.join('../exp/bam/mm2_trimmed/all/')
tmp_dir = '../tmp/fastq_IS_cover'
dat_exp_dir = '../exp/dat'
fig_dir = '../exp/fig/IS_cover'

os.makedirs(tmp_dir, exist_ok=True)
os.makedirs(dat_exp_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

In [3]:
ids = pd.read_csv(file_list_path)
ids.head()

,IS_Detect_ID,ParentLine,SubLine,gen,file_name,Anc,RecA,Prefix,sample_name_raw,Contig_Date,Complete,Folder,Folder_check,folder_err,File
0,NaN,0,0,Anc,20230904/MDS42_IS1.fa,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,L01_Anc,1,1,FACS,20231004/L01_Anc_m1.fasta,R01,1,L01-1,L01_Anc,20220418,True,20231004.0,20231004.0,NaN,L01_Anc_m1.fasta
2,L01_Anc,1,2,FACS,20231004/L01_Anc_m1.fasta,R01,1,L01-2,L01_Anc,20220418,True,20231004.0,20231004.0,NaN,L01_Anc_m1.fasta
3,L01_Anc,1,3,FACS,20231004/L01_Anc_m1.fasta,R01,1,L01-3,L01_Anc,20220418,True,20231004.0,20231004.0,NaN,L01_Anc_m1.fasta
4,L01_Anc,1,4,FACS,20231004/L01_Anc_m1.fasta,R01,1,L01-4,L01_Anc,20220418,True,20231004.0,20231004.0,NaN,L01_Anc_m1.fasta


In [4]:
ids.gen.unique()

array(['Anc', 'FACS', '8', '20'], dtype=object)

In [5]:
ids = ids.query('gen != "Anc"')
ids['gen_id'] = ids.apply(lambda x: 1 if x.gen == 'FACS' else 2 if x.gen == '8' else 3, axis=1)
ids['sample'] = ids.IS_Detect_ID
ids = ids.drop_duplicates(subset='sample').reset_index(drop=True)
ids.tail()

,IS_Detect_ID,ParentLine,SubLine,gen,file_name,Anc,RecA,Prefix,sample_name_raw,Contig_Date,Complete,Folder,Folder_check,folder_err,File,gen_id,sample
94,L10-4_G20,10,4,20,20231003/L10-4_G20.fasta,r04,0,L10-4,L10-4_G20,20230909,True,20231003.0,20231003.0,NaN,L10-4_G20.fasta,3,L10-4_G20
95,L11-1_G20,11,1,20,20231003/L11-1_G20.fasta,r05,0,L11-1,L11-1_G20,20230909,True,20231003.0,20231003.0,NaN,L11-1_G20.fasta,3,L11-1_G20
96,L11-2_G20,11,2,20,20250116/L11-2_G20.fasta,r05,0,L11-2,L11-2_G20,20230909,True,20250116.0,20250116.0,NaN,L11-2_G20.fasta,3,L11-2_G20
97,L11-3_G20,11,3,20,20231003/L11-3_G20.fasta,r05,0,L11-3,L11-3_G20,20230909,True,20231003.0,20231003.0,NaN,L11-3_G20.fasta,3,L11-3_G20
98,L11-4_G20,11,4,20,20231003/L11-4_G20.fasta,r05,0,L11-4,L11-4_G20,20230909,False,20231003.0,20231003.0,NaN,L11-4_G20.fasta,3,L11-4_G20


In [6]:
is_pos_df = pd.read_csv(is_pos_path)
is_pos_df.head()

,start,end,IS_strand,max_alignment_length,length,cluster_id,Line,Gen
0,476928,480020,reverse,3093,3093,0,L01-1,1
1,557167,560259,reverse,3093,3093,1,L01-1,1
2,1187831,1192502,forward,3093,4672,2,L01-1,1
3,1405123,1408215,forward,3093,3093,3,L01-1,1
4,1499230,1503892,reverse,3093,4663,4,L01-1,1


# Test with one genome 
## Test with one IS

In [7]:
def has_significant_gaps_or_insertions(cigar, threshold):
    for op, length in cigar:
		# 1 = insertion, 2 = deletion
        if op in (1, 2) and length > threshold:  
            return True
    return False

In [8]:
sample = 'L01-1_G20'
Line, Gen = sample.split('_')
if Gen == 'Anc':
	Gen = 1
	Line = Line+'-1'
elif Gen == 'G08':
	Gen = 2
else:
	Gen = 3
is_pos_df_sample = is_pos_df[(is_pos_df.Line == Line) & (is_pos_df.Gen == Gen)]
is_pos_df_sample.head()

,start,end,IS_strand,max_alignment_length,length,cluster_id,Line,Gen
27,476924,480016,reverse,3093,3093,0,L01-1,3
28,552064,555156,reverse,3093,3093,1,L01-1,3
29,706058,709150,reverse,3093,3093,2,L01-1,3
30,1062227,1065319,forward,3093,3093,3,L01-1,3
31,1188620,1193279,forward,3092,4660,4,L01-1,3


In [9]:
# Parameters
min_quality = 30          # Minimum mapping quality
threshold = 10            # Maximum gap/insert size allowed
buffer = 100

# Parse the region
is_cluster_id = 2
bam_file = os.path.join(mm2_output_dir, f'{sample}.bam')
chrom = sample 
start, end = is_pos_df_sample[is_pos_df_sample.cluster_id == is_cluster_id][['start', 'end']].values[0]

# Open the BAM file
bam = pysam.AlignmentFile(bam_file, "rb")

# Count reads that completely cover the region without significant gaps/insertions
valid_reads = []
read_cnt = 0
for read in bam.fetch(chrom, start, end):
    read_cnt += 1
    if (
        read.mapping_quality >= min_quality  # Minimum quality filter
        and not read.is_unmapped            # Ignore unmapped reads
        and not read.is_secondary           # Ignore secondary alignments
        and read.reference_start <= start - buffer  # Start of read covers the region start
        and read.reference_end >= end + buffer     # End of read covers the region end
        and not has_significant_gaps_or_insertions(read.cigartuples, threshold)  # Check gaps
    ):
        valid_reads.append(read)

read_count = len(valid_reads)

# Output the result
print(f"Number of reads completely covering the region: {read_count}")
print(f"Total number of reads: {read_cnt}")

Number of reads completely covering the region: 9
Total number of reads: 26


In [10]:
valid_read_data = []
for read in valid_reads:
    valid_read_data.append({
        "read_name": read.query_name,          # Read name
        "chromosome": read.reference_name,    # Reference chromosome
        "start": read.reference_start,        # Alignment start
        "end": read.reference_end,            # Alignment end
        "mapping_quality": read.mapping_quality,  # Mapping quality
        "cigar": read.cigarstring,             # CIGAR string
		"is_cluster_id": is_cluster_id
    })

# Convert to Pandas DataFrame
reads_df = pd.DataFrame(valid_read_data)
reads_df.head()

,read_name,chromosome,start,end,mapping_quality,cigar,is_cluster_id
0,2af1a467-9f2e-4925-9eaa-20207dcb0fd3,L01-1_G20,693507,716158,60,22M1I26M2I70M3D10M1D1M1D20M1I4M1I47M1D254M1I1M...,2
1,4c5e5ad9-1ede-4fb0-ac91-16d30a308afc,L01-1_G20,695178,719516,60,183M3D29M3D120M1D2M1D43M1D84M2D60M1I39M2D2M2I3...,2
2,09ecd766-a3d5-41d9-ad30-8f3c7b945851,L01-1_G20,695673,726410,60,6M1I339M1D524M3D46M2D9M1D247M1I92M1I1M2D39M3I1...,2
3,50cacd0a-a88a-498b-80d9-4fbaa52905eb,L01-1_G20,698140,731518,60,7M1I31M1D164M1D196M1D5M1D213M1D98M1D79M3D58M2D...,2
4,941d49d5-df3b-4bec-9795-10a260ea3842,L01-1_G20,698787,712049,60,19M2D4M1I350M1D2M1D8M2D3M1I88M3D5M1D103M3D74M2...,2


In [11]:
# make function
def read_cover_stat(bam, chrom, start, end, min_quality=30, threshold=10, buffer=100):
	valid_reads = []
	read_cnt = 0
	for read in bam.fetch(chrom, start, end):
		read_cnt += 1
		if (
			read.mapping_quality >= min_quality  # Minimum quality filter
			and not read.is_unmapped            # Ignore unmapped reads
			and not read.is_secondary           # Ignore secondary alignments
			and read.reference_start <= start - buffer  # Start of read covers the region start
			and read.reference_end >= end + buffer     # End of read covers the region end
			and not has_significant_gaps_or_insertions(read.cigartuples, threshold)  # Check gaps
		):
			valid_reads.append(read)
	
	valid_read_data = []
	for read in valid_reads:
		valid_read_data.append({
			"read_name": read.query_name,          # Read name
			"chromosome": read.reference_name,    # Reference chromosome
			"start": read.reference_start,        # Alignment start
			"end": read.reference_end,            # Alignment end
			"mapping_quality": read.mapping_quality,  # Mapping quality
			"cigar": read.cigarstring,             # CIGAR string
		})
	return pd.DataFrame(valid_read_data), read_cnt

# example
read_cover_stat(bam, chrom, start, end)

(                              read_name chromosome   start     end  \
 0  2af1a467-9f2e-4925-9eaa-20207dcb0fd3  L01-1_G20  693507  716158   
 1  4c5e5ad9-1ede-4fb0-ac91-16d30a308afc  L01-1_G20  695178  719516   
 2  09ecd766-a3d5-41d9-ad30-8f3c7b945851  L01-1_G20  695673  726410   
 3  50cacd0a-a88a-498b-80d9-4fbaa52905eb  L01-1_G20  698140  731518   
 4  941d49d5-df3b-4bec-9795-10a260ea3842  L01-1_G20  698787  712049   
 5  3776cd62-fb49-4e49-afc7-80b6e1955078  L01-1_G20  702570  725295   
 6  60cdf24f-88ce-43d7-ae15-750789eded9d  L01-1_G20  702846  738449   
 7  192807a7-fd4b-4e10-ac48-2ff9e4c45a69  L01-1_G20  703391  721454   
 8  f204bd5c-350a-429a-b66e-aa484572b34c  L01-1_G20  704859  717439   
 
    mapping_quality                                              cigar  
 0               60  22M1I26M2I70M3D10M1D1M1D20M1I4M1I47M1D254M1I1M...  
 1               60  183M3D29M3D120M1D2M1D43M1D84M2D60M1I39M2D2M2I3...  
 2               60  6M1I339M1D524M3D46M2D9M1D247M1I92M1I1M2D39M3I1..

## run for all ISs

In [12]:
def has_significant_gaps_or_insertions_in_range(cigar, threshold, region_start, region_end, read_start, debug = False):
	ref_pos = read_start  # Tracks reference position

	for op, length in cigar:
		if op == 0:  # Match
			ref_pos += length

		elif op == 1:  # Insertion
			# Check if insertion falls within the specified region
			if region_start <= ref_pos <= region_end and length > threshold:
				if debug:
					print(f'ins: {op}, {length}, {ref_pos}, {region_start}, {region_end}')
				return True

		elif op == 2:  # Deletion
			# Check if the deletion overlaps with the specified region
			deletion_start = ref_pos
			deletion_end = ref_pos + length
			overlap_length = max(0, min(deletion_end, region_end) - max(deletion_start, region_start))
			if debug:
				print(f'deletion_start: {deletion_start}, deletion_end: {deletion_end}, region_start: {region_start}, region_end: {region_end}', overlap_length)
			if (
				deletion_end >= region_start and  # Deletion overlaps region start
				deletion_start <= region_end and  # Deletion overlaps region end
				overlap_length > threshold
			):
				return True
			ref_pos += length  # Update reference position

		elif op in (3, 4, 5):  # Skip, soft clip, hard clip, etc.
			# Adjust ref_pos for skipped regions but ignore them for gap checks
			if op == 3:  # Skipped region (N), thogh unlikely as not included in the option
				ref_pos += length
			# not necessary as clips are found in the ends, but just in case.
			elif region_start <= ref_pos <= region_end:
				if debug:
					print(f'clip: {op}, {length}, {ref_pos}')
				return True
		if ref_pos > region_end:
			break
	return False

def read_cover_stat(bam, chrom, start, end, min_quality=30, threshold=10, buffer=100, start_inside = 0, debug = False):
	valid_reads = []
	read_cnt = 0
	for read in bam.fetch(chrom, start, end):
		if debug:
			print(f'read: {read} -------------------')
			print(f'{read.reference_start} -- {read.reference_end}, {read.mapping_quality}, {read.is_secondary}, {read.is_unmapped}')

		read_cnt += 1
		if (
			read.mapping_quality >= min_quality  # Minimum quality filter
			and not read.is_unmapped            # Ignore unmapped reads
			and not read.is_secondary           # Ignore secondary alignments
			and read.reference_start <= start - buffer  # Start of read covers the region start
			and read.reference_end >= end + buffer     # End of read covers the region end
			and not has_significant_gaps_or_insertions_in_range(read.cigartuples, threshold,
							start+start_inside, end-start_inside, read.reference_start, debug = debug) 
		):
			valid_reads.append(read)
	
	valid_read_data = []
	for read in valid_reads:
		valid_read_data.append({
			"read_name": read.query_name,          # Read name
			"chromosome": read.reference_name,    # Reference chromosome
			"start": read.reference_start,        # Alignment start
			"end": read.reference_end,            # Alignment end
			"mapping_quality": read.mapping_quality,  # Mapping quality
			"cigar": read.cigarstring,             # CIGAR string
		})
	return pd.DataFrame(valid_read_data), read_cnt

In [13]:
sample = 'L05-4_G08'
bam_file = os.path.join(mm2_output_dir, f'{sample}.bam')
bam = pysam.AlignmentFile(bam_file, "rb")
#1709698-1712790
read_cover_stat(bam, sample, 1709698, 1712790, buffer = 0, threshold=10, debug = True)[0]

read: 4c71ad56-4119-4fbd-b278-342ebd6b1891	0	#0	1674592	60	3495S36M1D25M1D82M3D12M1D60M1D124M1I12M1I30M1I14M3D43M2I49M3I17M1I2M1D80M1D7M2I59M1D11M1D76M1I6M2D9M1D35M1D74M2I32M3I80M1D9M2D97M1I9M2D4M5I1M2D83M3I122M1I14M2D26M1D7M1D1M2D60M1I431M1I2M3D10M1D6M3D40M1I15M1D73M1I24M3I55M4D1M3D12M2D20M1D32M6D13M2D30M1I196M1I72M1I60M1D62M1D127M6D5M1I16M1I103M1D58M3D96M1I126M1I74M1D8M2D53M6I32M1D2M2I137M3D6M2I19M2D6M1I11M3D3M3D24M2D7M1I53M2D31M2D5M1D2M1D5M1I5M1I8M2D1M2D7M1I1M1I6M3D57M1D40M2I25M2D6M1I2M3D66M2D4M2D79M2I9M2I6M1I15M2I3M1D6M1I2M2D12M1D25M1D7M1D7M3D17M1I7M1I68M2D106M1D56M1I15M1D4M1D19M2I107M4I96M3I86M1D114M1D3M1D29M2I31M1I30M1I3M1I22M1I8M1D65M1I14M1D14M2I51M2D5M1D68M1I4M1D103M1I29M1I1M1I35M1D78M2D19M2D7M2D2M1D73M1I8M1D6M1D124M1I3M2D5M1I270M1I22M1D8M1D16M2D120M1I29M2I3M1I3M3D49M2D3M1D10M1D20M1D15M1I150M1D20M1I7M1D36M1D58M1I39M1I5M1D46M2D58M1I6M1I55M2D1M1D27M1D136M2D105M1D74M3I3M1D212M6D158M2I1M1D8M1D10M1D144M1D32M2D188M4I12M3D94M2D295M2D47M2D97M3D24M2D76M3D147M2D3M1D42M1D94M1D45M1I87M1D16

,read_name,chromosome,start,end,mapping_quality,cigar
0,4c71ad56-4119-4fbd-b278-342ebd6b1891,L05-4_G08,1674591,1723368,60,3495S36M1D25M1D82M3D12M1D60M1D124M1I12M1I30M1I...
1,96077a40-2b39-4628-a7cd-aa6284b3751d,L05-4_G08,1705005,1720380,60,45M1I10M1D53M1D45M1D50M2D64M1I124M1I12M1D14M2I...
2,4b16095f-910e-4eef-9c5a-66da2e07c99c,L05-4_G08,1709695,1723811,60,610S41M2I13M1I51M1D336M2D25M3I476M3D4M2D59M1D1...
3,5303f1bc-93be-4fec-a221-8daed13008d5,L05-4_G08,1709695,1716254,60,14645H27M2I19M1I4M1I3M1I12M1I182M3D31M1D11M3D3...
4,0aada627-a8cd-45df-bd80-8b17b08e1af1,L05-4_G08,1709695,1714054,60,5694H114M1I5M2D16M1I23M7I128M1D1M1D2M1D2M2I7M3...
5,abe4007b-2bc4-4638-bd17-bac3d63a3655,L05-4_G08,1709696,1714581,60,1230S14M2D2M3D18M1D70M1I25M5I50M1I6M1I26M1I37M...


In [14]:
is_stat_rows = []  # Store rows as dictionaries for the main DataFrame
os.makedirs(os.path.join(tmp_dir, 'is_cover_stat'), exist_ok=True)

for row in ids.itertuples():
    sample = row.sample

    # Subset to the specific FASTA
    Line, Gen = sample.split('_')
    if Gen == 'Anc':
        Gen = 1
        Line = Line + '-1' # check only subline 1
    elif Gen == 'G08':
        Gen = 2
    else:
        Gen = 3
    is_pos_df_sample = is_pos_df[(is_pos_df.Line == Line) & (is_pos_df.Gen == Gen)]

    bam_file = os.path.join(mm2_output_dir, f'{sample}.bam')
    bam = pysam.AlignmentFile(bam_file, "rb")

    reads_rows = []  # Store rows for the reads DataFrame
    for is_cluster_id in is_pos_df_sample.cluster_id:
        start, end = is_pos_df_sample[is_pos_df_sample.cluster_id == is_cluster_id][['start', 'end']].values[0]
        # to avoid removing multimapped reads in alignment, set min_quality=0
        reads_df, read_cnt = read_cover_stat(bam, sample, start, end, min_quality=0, threshold=10)
        reads_df['Sample'] = sample
        reads_df['is_cluster_id'] = is_cluster_id
        reads_df['read_cnt'] = read_cnt

        # Append rows as dictionaries
        reads_rows.extend(reads_df.to_dict('records'))  # Flatten to a list of dicts
        is_stat_rows.append({
            'Sample': sample,
            'is_cluster_id': is_cluster_id,
            'read_cnt': read_cnt,
            'cover_cnt': reads_df.shape[0],
            'Line': Line,
            'Gen': Gen
        })

    # Create and save the reads DataFrame for this sample
    reads_df_sample = pd.DataFrame(reads_rows)
    reads_df_sample.to_pickle(os.path.join(tmp_dir, 'is_cover_stat', f'{sample}.pkl'))
    print(f"Processed {sample}")

# Final DataFrame for `is_stat_df`
is_stat_df = pd.DataFrame(is_stat_rows)
is_stat_df.head()

Processed L01_Anc
Processed L02_Anc
Processed L03_Anc
Processed L04_Anc
Processed L05_Anc
Processed L06_Anc
Processed L07_Anc
Processed L08_Anc
Processed L09_Anc
Processed L10_Anc
Processed L11_Anc
Processed L01-1_G08
Processed L01-2_G08
Processed L01-3_G08
Processed L01-4_G08
Processed L02-1_G08
Processed L02-2_G08
Processed L02-3_G08
Processed L02-4_G08
Processed L03-1_G08
Processed L03-2_G08
Processed L03-3_G08
Processed L03-4_G08
Processed L04-1_G08
Processed L04-2_G08
Processed L04-3_G08
Processed L04-4_G08
Processed L05-1_G08
Processed L05-2_G08
Processed L05-3_G08
Processed L05-4_G08
Processed L06-1_G08
Processed L06-2_G08
Processed L06-3_G08
Processed L06-4_G08
Processed L07-1_G08
Processed L07-2_G08
Processed L07-3_G08
Processed L07-4_G08
Processed L08-1_G08
Processed L08-2_G08
Processed L08-3_G08
Processed L08-4_G08
Processed L09-1_G08
Processed L09-2_G08
Processed L09-3_G08
Processed L09-4_G08
Processed L10-1_G08
Processed L10-2_G08
Processed L10-3_G08
Processed L10-4_G08
Pr

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen
0,L01_Anc,0,32,21,L01-1,1
1,L01_Anc,1,18,10,L01-1,1
2,L01_Anc,2,11,0,L01-1,1
3,L01_Anc,3,24,10,L01-1,1
4,L01_Anc,4,25,0,L01-1,1


In [15]:
is_stat_df_merge = is_stat_df.merge(is_pos_df,
	left_on=['Line', 'Gen', 'is_cluster_id'],
	right_on=['Line', 'Gen', 'cluster_id'],
	how='left')
is_stat_df_merge.to_csv(os.path.join(dat_exp_dir, 'is_cover_stat.csv'), index=False)
is_stat_df_merge.head()

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
0,L01_Anc,0,32,21,L01-1,1,476928,480020,reverse,3093,3093,0
1,L01_Anc,1,18,10,L01-1,1,557167,560259,reverse,3093,3093,1
2,L01_Anc,2,11,0,L01-1,1,1187831,1192502,forward,3093,4672,2
3,L01_Anc,3,24,10,L01-1,1,1405123,1408215,forward,3093,3093,3
4,L01_Anc,4,25,0,L01-1,1,1499230,1503892,reverse,3093,4663,4


In [16]:
# All ISs are covered by atleast three reads
is_stat_df_merge.query('Gen > 1 & cover_cnt < 5')

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
152,L01-2_G08,4,12,4,L01-2,2,1170698,1173790,reverse,3093,3093,4
174,L01-3_G08,10,18,4,L01-3,2,1922673,1925768,reverse,3096,3096,10
178,L01-3_G08,14,16,4,L01-3,2,2813531,2816622,reverse,3093,3092,14
474,L05-1_G08,6,14,4,L05-1,2,1876641,1879732,forward,3093,3092,6
477,L05-1_G08,9,17,4,L05-1,2,2072811,2075903,forward,3093,3093,9
526,L05-3_G08,16,10,4,L05-3,2,3392631,3395723,reverse,3093,3093,16
538,L05-4_G08,9,16,2,L05-4,2,1709698,1712790,reverse,3093,3093,9
539,L05-4_G08,10,22,3,L05-4,2,1720277,1723369,forward,3093,3093,10
544,L05-4_G08,15,10,4,L05-4,2,2452426,2455522,forward,3097,3097,15
583,L06-2_G08,13,9,4,L06-2,2,2600285,2603378,forward,3094,3094,13


In [17]:
# Some Anc ISs have poor reads (as expected, positive control)
is_stat_df_merge.query('Gen > 0 & cover_cnt < 2')

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
2,L01_Anc,2,11,0,L01-1,1,1187831,1192502,forward,3093,4672,2
4,L01_Anc,4,25,0,L01-1,1,1499230,1503892,reverse,3093,4663,4
29,L03_Anc,4,58,0,L03-1,1,1872986,1878156,reverse,3093,5171,4
47,L04_Anc,13,34,0,L04-1,1,3626825,3629917,forward,3093,3093,13
51,L05_Anc,3,142,1,L05-1,1,1859371,1862461,reverse,3092,3091,3
54,L05_Anc,6,65,0,L05-1,1,2525641,2528732,forward,3092,3092,6
67,L06_Anc,9,26,0,L06-1,1,3725465,3728556,reverse,3093,3092,9
101,L09_Anc,4,24,0,L09-1,1,1697666,1701755,reverse,3094,4090,4
102,L09_Anc,5,33,0,L09-1,1,1752850,1756937,reverse,3093,4088,5


In [18]:
# even large ISs have good coverage
is_stat_df_merge.sort_values('length', ascending=False).head()

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
1857,L10-2_G20,19,33,9,L10-2,3,3739423,3752842,forward,3093,13420,19
855,L10-2_G08,20,23,5,L10-2,2,3766881,3780299,forward,3094,13419,20
248,L02-2_G08,22,65,20,L02-2,2,3889503,3899446,forward,3093,9944,22
1899,L10-4_G20,19,21,6,L10-4,3,3375937,3385258,reverse,3097,9322,19
1877,L10-3_G20,18,59,13,L10-3,3,3771459,3780755,reverse,3093,9297,18


In [19]:
is_stat_df_merge_buffer = pd.read_csv(os.path.join(dat_exp_dir, 'is_cover_stat.csv'))

## Without buffer sequence

In [20]:
is_stat_rows = []  # Store rows as dictionaries for the main DataFrame
os.makedirs(os.path.join(tmp_dir, 'is_cover_stat'), exist_ok=True)

for row in ids.itertuples():
    sample = row.sample

    # Subset to the specific FASTA
    Line, Gen = sample.split('_')
    if Gen == 'Anc':
        Gen = 1
        Line = Line + '-1' # check only subline 1
    elif Gen == 'G08':
        Gen = 2
    else:
        Gen = 3
    is_pos_df_sample = is_pos_df[(is_pos_df.Line == Line) & (is_pos_df.Gen == Gen)]

    bam_file = os.path.join(mm2_output_dir, f'{sample}.bam')
    bam = pysam.AlignmentFile(bam_file, "rb")

    reads_rows = []  # Store rows for the reads DataFrame
    for is_cluster_id in is_pos_df_sample.cluster_id:
        start, end = is_pos_df_sample[is_pos_df_sample.cluster_id == is_cluster_id][['start', 'end']].values[0]
        # to avoid removing multimapped reads in alignment, set min_quality=0
        reads_df, read_cnt = read_cover_stat(bam, sample, start, end, min_quality=0, threshold=10, buffer = 0)
        reads_df['Sample'] = sample
        reads_df['is_cluster_id'] = is_cluster_id
        reads_df['read_cnt'] = read_cnt

        # Append rows as dictionaries
        reads_rows.extend(reads_df.to_dict('records'))  # Flatten to a list of dicts
        is_stat_rows.append({
            'Sample': sample,
            'is_cluster_id': is_cluster_id,
            'read_cnt': read_cnt,
            'cover_cnt': reads_df.shape[0],
            'Line': Line,
            'Gen': Gen
        })

    # Create and save the reads DataFrame for this sample
    reads_df_sample = pd.DataFrame(reads_rows)
    reads_df_sample.to_pickle(os.path.join(tmp_dir, 'is_cover_stat', f'{sample}.pkl'))
    print(f"Processed {sample}")

# Final DataFrame for `is_stat_df`
is_stat_df = pd.DataFrame(is_stat_rows)
is_stat_df.head()

Processed L01_Anc
Processed L02_Anc
Processed L03_Anc
Processed L04_Anc
Processed L05_Anc
Processed L06_Anc
Processed L07_Anc
Processed L08_Anc
Processed L09_Anc
Processed L10_Anc
Processed L11_Anc
Processed L01-1_G08
Processed L01-2_G08
Processed L01-3_G08
Processed L01-4_G08
Processed L02-1_G08
Processed L02-2_G08
Processed L02-3_G08
Processed L02-4_G08
Processed L03-1_G08
Processed L03-2_G08
Processed L03-3_G08
Processed L03-4_G08
Processed L04-1_G08
Processed L04-2_G08
Processed L04-3_G08
Processed L04-4_G08
Processed L05-1_G08
Processed L05-2_G08
Processed L05-3_G08
Processed L05-4_G08
Processed L06-1_G08
Processed L06-2_G08
Processed L06-3_G08
Processed L06-4_G08
Processed L07-1_G08
Processed L07-2_G08
Processed L07-3_G08
Processed L07-4_G08
Processed L08-1_G08
Processed L08-2_G08
Processed L08-3_G08
Processed L08-4_G08
Processed L09-1_G08
Processed L09-2_G08
Processed L09-3_G08
Processed L09-4_G08
Processed L10-1_G08
Processed L10-2_G08
Processed L10-3_G08
Processed L10-4_G08
Pr

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen
0,L01_Anc,0,32,21,L01-1,1
1,L01_Anc,1,18,11,L01-1,1
2,L01_Anc,2,11,7,L01-1,1
3,L01_Anc,3,24,10,L01-1,1
4,L01_Anc,4,25,16,L01-1,1


In [21]:
is_stat_df_merge = is_stat_df.merge(is_pos_df,
	left_on=['Line', 'Gen', 'is_cluster_id'],
	right_on=['Line', 'Gen', 'cluster_id'],
	how='left')
is_stat_df_merge.to_csv(os.path.join(dat_exp_dir, 'is_cover_stat_no_buff.csv'), index=False)
is_stat_df_merge.head()

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
0,L01_Anc,0,32,21,L01-1,1,476928,480020,reverse,3093,3093,0
1,L01_Anc,1,18,11,L01-1,1,557167,560259,reverse,3093,3093,1
2,L01_Anc,2,11,7,L01-1,1,1187831,1192502,forward,3093,4672,2
3,L01_Anc,3,24,10,L01-1,1,1405123,1408215,forward,3093,3093,3
4,L01_Anc,4,25,16,L01-1,1,1499230,1503892,reverse,3093,4663,4


In [22]:
# All ISs are covered by atleast three reads
is_stat_df_merge.query('Gen > 1 & cover_cnt < 5')

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
178,L01-3_G08,14,16,4,L01-3,2,2813531,2816622,reverse,3093,3092,14
526,L05-3_G08,16,10,4,L05-3,2,3392631,3395723,reverse,3093,3093,16
544,L05-4_G08,15,10,4,L05-4,2,2452426,2455522,forward,3097,3097,15
583,L06-2_G08,13,9,4,L06-2,2,2600285,2603378,forward,3094,3094,13
584,L06-2_G08,14,10,3,L06-2,2,3015186,3018277,reverse,3094,3092,14
586,L06-2_G08,16,12,3,L06-2,2,3258472,3261562,forward,3092,3091,16
595,L06-2_G08,25,19,3,L06-2,2,3971299,3974391,reverse,3093,3093,25
844,L10-2_G08,9,10,4,L10-2,2,1949123,1952215,forward,3094,3093,9
1085,L02-1_G20,29,13,4,L02-1,3,4048471,4051562,forward,3093,3092,29
1156,L02-4_G20,12,10,4,L02-4,3,2317190,2320279,reverse,3092,3090,12


In [23]:
# Some Anc ISs have poor reads (as expected, positive control)
is_stat_df_merge.query('Gen > 0 & cover_cnt < 2')

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
29,L03_Anc,4,58,0,L03-1,1,1872986,1878156,reverse,3093,5171,4
47,L04_Anc,13,34,0,L04-1,1,3626825,3629917,forward,3093,3093,13
54,L05_Anc,6,65,1,L05-1,1,2525641,2528732,forward,3092,3092,6
101,L09_Anc,4,24,0,L09-1,1,1697666,1701755,reverse,3094,4090,4
102,L09_Anc,5,33,0,L09-1,1,1752850,1756937,reverse,3093,4088,5


In [24]:
# even large ISs have good coverage
is_stat_df_merge.sort_values('length', ascending=False).head()

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
1857,L10-2_G20,19,33,9,L10-2,3,3739423,3752842,forward,3093,13420,19
855,L10-2_G08,20,23,5,L10-2,2,3766881,3780299,forward,3094,13419,20
248,L02-2_G08,22,65,22,L02-2,2,3889503,3899446,forward,3093,9944,22
1899,L10-4_G20,19,21,6,L10-4,3,3375937,3385258,reverse,3097,9322,19
1877,L10-3_G20,18,59,15,L10-3,3,3771459,3780755,reverse,3093,9297,18


In [25]:
is_pos_df.head()

,start,end,IS_strand,max_alignment_length,length,cluster_id,Line,Gen
0,476928,480020,reverse,3093,3093,0,L01-1,1
1,557167,560259,reverse,3093,3093,1,L01-1,1
2,1187831,1192502,forward,3093,4672,2,L01-1,1
3,1405123,1408215,forward,3093,3093,3,L01-1,1
4,1499230,1503892,reverse,3093,4663,4,L01-1,1


## create batch script

In [26]:
is_lst = ['L10-3.1.8', 'L10-3.2.12', 'L10-3.2.2', 'L10-3.2.8', 'L10-3.3.3']
range_size = 20000

# Open the batch file for writing
bat_file_dir = os.path.join(bat_dir, 'L10-3_long.bat')
with open(bat_file_dir, "w") as bat_file:
	# Write the IGV setup commands
	bat_file.write(f'snapshotDirectory {igv_snapshot_dir}\n')
	bat_file.write("preference DEFAULT_FONT_SIZE 20\n")

	for is_id in is_lst:
		# Convert IS ID to sample name
		line_ = is_id.split('.')[0]
		gen_ = is_id.split('.')[1]
		sample_ = line_ + ('_G08' if gen_ == '2' else '_G20')
		if gen_ == '1':
			sample_ = is_id.split('-')[0] + '_Anc'
		is_cluster_id_ = int(is_id.split('.')[2])

		# create bed file for IGV with strand
		bed_df = is_pos_df.query(f'Line == "{line_}" & Gen == {gen_}')
		bed_df['chr'] = sample_
		bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
		bed_df['IS_ID'] = bed_df.cluster_id.apply(lambda x: f'{line_}.{gen_}.{x}')
		bed_df['start'] = bed_df['start'] - 1
		bed_df = bed_df[['chr', 'start', 'end', 'IS_ID', 'length', 'strand']]
		bed_df.to_csv(os.path.join(bed_dir, f'{sample_}.bed'), sep='\t', header=False, index=False)

		# Query the dataframe to get the start and end coordinates
		row = is_stat_df_merge.query(f'Sample == "{sample_}" & is_cluster_id == {is_cluster_id_}').iloc[0]
		vis_range_mid = (row.start + row.end) // 2
		vis_range_start = max(0, vis_range_mid - range_size//2)
		vis_range_end = vis_range_start + range_size

		# Generate IGV commands
		bat_file.write("new\n")
		bat_file.write(f"genome {igv_fasta_dir}\\{sample_}.fasta\n")
		bat_file.write(f"load {igv_bam_dir}\\{sample_}.bam\n")
		bat_file.write(f"load {igv_bed_dir}\\{sample_}.bed\n")
		bat_file.write(f"goto {sample_}:{vis_range_start}-{vis_range_end}\n")
		#bat_file.write(f"collapse {sample_}.bam\n")
		bat_file.write(f"maxPanelHeight 600\n")
		bat_file.write(f"expand {sample_}.bam\n")
		bat_file.write(f"setColor 155,155,155 {sample_}.bam\n")
		#bat_file.write(f"setTrackHeight 1000 {sample_}.bam\n")
		bat_file.write(f"setTrackHeight 15 {sample_}.bed\n")
		bat_file.write(f"snapshot {is_id}.png\n")
		bat_file.write(f"snapshot {is_id}.svg\n")

		bat_file.write(f"\n\n")
	
	#bat_file.write("exit\n")  # Exit IGV when done

/tmp/ipykernel_19897/2436434166.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['chr'] = sample_
/tmp/ipykernel_19897/2436434166.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
/tmp/ipykernel_19897/2436434166.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

In [27]:
is_stat_df_merge_buffer.query('Gen > 1 & cover_cnt < 3')

,Sample,is_cluster_id,read_cnt,cover_cnt,Line,Gen,start,end,IS_strand,max_alignment_length,length,cluster_id
538,L05-4_G08,9,16,2,L05-4,2,1709698,1712790,reverse,3093,3093,9
1887,L10-4_G20,7,8,2,L10-4,3,1906752,1909847,forward,3102,3096,7


In [28]:
is_lst = ['L05-4.2.9', 'L04-3.2.9', 'L10-4.3.7']
is_stat_df_merge_buffer.query('Gen > 1 & cover_cnt < 3')
range_size = 20000

# Open the batch file for writing
bat_file_dir = os.path.join(bat_dir, 'poor_IS_cover.bat')
with open(bat_file_dir, "w") as bat_file:
	# Write the IGV setup commands
	bat_file.write(f'snapshotDirectory {igv_snapshot_dir}\n')
	bat_file.write("preference DEFAULT_FONT_SIZE 20\n")

	for is_id in is_lst:
		# Convert IS ID to sample name
		line_ = is_id.split('.')[0]
		gen_ = is_id.split('.')[1]
		sample_ = line_ + ('_G08' if gen_ == '2' else '_G20')
		if gen_ == '1':
			sample_ = is_id.split('-')[0] + '_Anc'
		is_cluster_id_ = int(is_id.split('.')[2])

		# create bed file for IGV with strand
		bed_df = is_pos_df.query(f'Line == "{line_}" & Gen == {gen_}')
		bed_df['chr'] = sample_
		bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
		bed_df['IS_ID'] = bed_df.cluster_id.apply(lambda x: f'{line_}.{gen_}.{x}')
		bed_df['start'] = bed_df['start'] - 1
		bed_df = bed_df[['chr', 'start', 'end', 'IS_ID', 'length', 'strand']]
		bed_df.to_csv(os.path.join(bed_dir, f'{sample_}.bed'), sep='\t', header=False, index=False)

		# Query the dataframe to get the start and end coordinates
		row = is_stat_df_merge.query(f'Sample == "{sample_}" & is_cluster_id == {is_cluster_id_}').iloc[0]
		vis_range_mid = (row.start + row.end) // 2
		vis_range_start = max(0, vis_range_mid - range_size//2)
		vis_range_end = vis_range_start + range_size

		# Generate IGV commands
		bat_file.write("new\n")
		bat_file.write(f"genome {igv_fasta_dir}\\{sample_}.fasta\n")
		bat_file.write(f"load {igv_bam_dir}\\{sample_}.bam\n")
		bat_file.write(f"load {igv_bed_dir}\\{sample_}.bed\n")
		bat_file.write(f"goto {sample_}:{vis_range_start}-{vis_range_end}\n")
		#bat_file.write(f"collapse {sample_}.bam\n")
		bat_file.write(f"maxPanelHeight 600\n")
		bat_file.write(f"expand {sample_}.bam\n")
		bat_file.write(f"setColor 155,155,155 {sample_}.bam\n")
		#bat_file.write(f"setTrackHeight 1000 {sample_}.bam\n")
		bat_file.write(f"setTrackHeight 15 {sample_}.bed\n")
		bat_file.write(f"snapshot {is_id}.png\n")
		bat_file.write(f"snapshot {is_id}.svg\n")

		bat_file.write(f"\n\n")
	
	#bat_file.write("exit\n")  # Exit IGV when done

/tmp/ipykernel_19897/1179406447.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['chr'] = sample_
/tmp/ipykernel_19897/1179406447.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['strand'] = bed_df.IS_strand.apply(lambda x: '+' if x == 'forward' else '-')
/tmp/ipykernel_19897/1179406447.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

after running IGV batch run
```bash
mkdir -p highres  
ls *.svg | xargs -I {} sh -c 'inkscape "{}" -o "highres/$(basename "{}" .svg).png" -d 200 -D'
```
This gives figures with ~ 700 kbp